# 10 - AdamW 优化器 (AI Infra 视角)

本节从 **工程实现** 角度理解 AdamW，重点关注：
- 显存占用分析
- 参数分组策略
- 分布式优化器 (ZeRO)
- 学习率调度

> 算法细节请参考原论文，这里只讲你需要知道的工程要点。

In [1]:
import torch
import torch.nn as nn
import math

## 1. AdamW 核心概念 (30秒版)

```
AdamW = Adam + Decoupled Weight Decay

每个参数维护两个状态:
  m (一阶动量): 梯度的滑动平均，平滑更新方向
  v (二阶动量): 梯度平方的滑动平均，自适应学习率

核心公式:
  m = β1 * m + (1-β1) * grad
  v = β2 * v + (1-β2) * grad²
  param = param - lr * m / √v - lr * wd * param
                  ↑ Adam 部分    ↑ 解耦的 weight decay
```

**关键点**: Weight decay 直接作用于参数，不污染动量。

## 2. 显存占用分析 (重要!)

这是面试高频考点。

### 混合精度训练的显存组成

| 组件 | 精度 | 大小 | 说明 |
|------|------|------|------|
| 模型参数 | FP16/BF16 | 2Φ | Φ = 参数量 |
| 梯度 | FP16/BF16 | 2Φ | 和参数同精度 |
| 优化器状态 m | FP32 | 4Φ | 一阶动量 |
| 优化器状态 v | FP32 | 4Φ | 二阶动量 |
| 主参数副本 | FP32 | 4Φ | 用于精确更新 |
| **总计** | | **16Φ** | |

### 实例计算

In [2]:
def estimate_training_memory(num_params_billion, batch_size, seq_len, hidden_dim, num_layers):
    """
    估算训练显存占用
    """
    num_params = num_params_billion * 1e9
    
    # 模型 + 优化器状态 (混合精度)
    model_mem = 2 * num_params  # FP16 参数
    grad_mem = 2 * num_params   # FP16 梯度
    optimizer_mem = 12 * num_params  # FP32: m + v + master weights
    
    # 激活值 (粗略估计，取决于是否用 activation checkpointing)
    activation_mem = batch_size * seq_len * hidden_dim * num_layers * 2 * 4  # 估计
    
    total = model_mem + grad_mem + optimizer_mem + activation_mem
    
    print(f"模型参数: {num_params_billion}B")
    print(f"模型权重: {model_mem / 1e9:.1f} GB")
    print(f"梯度: {grad_mem / 1e9:.1f} GB")
    print(f"优化器状态: {optimizer_mem / 1e9:.1f} GB")
    print(f"激活值 (估计): {activation_mem / 1e9:.1f} GB")
    print(f"总计: {total / 1e9:.1f} GB")
    return total

# 7B 模型
estimate_training_memory(7, batch_size=4, seq_len=2048, hidden_dim=4096, num_layers=32)

模型参数: 7B
模型权重: 14.0 GB
梯度: 14.0 GB
优化器状态: 84.0 GB
激活值 (估计): 8.6 GB
总计: 120.6 GB


120589934592.0

### 显存优化策略

| 策略 | 优化内容 | 节省比例 |
|------|----------|----------|
| ZeRO-1 | 分片优化器状态 | ~4x |
| ZeRO-2 | + 分片梯度 | ~8x |
| ZeRO-3 | + 分片参数 | ~Nx (N=GPU数) |
| Gradient Checkpointing | 激活值 | ~sqrt(layers) |
| Offload | CPU/NVMe | 极大但慢 |

## 3. 参数分组 (Weight Decay 策略)

**不是所有参数都应该加 weight decay!**

```
需要 decay:     矩阵权重 (Linear, Conv)
不需要 decay:   Bias, LayerNorm/RMSNorm, Embedding
```

### 原因
- **Bias**: 维度小，正则化收益低
- **Norm 参数**: 是缩放因子，衰减会破坏归一化效果  
- **Embedding**: 某些词可能出现频率低，衰减会让它们学不好

In [3]:
def configure_optimizers(model, lr, weight_decay):
    """
    nanochat 风格的参数分组
    
    简单规则: 1D 参数不加 decay，多维参数加 decay
    """
    decay_params = []
    no_decay_params = []
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        
        # 1D 参数 (bias, norm) 不加 weight decay
        if param.ndim == 1:
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    
    param_groups = [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0},
    ]
    
    return torch.optim.AdamW(param_groups, lr=lr)

# 测试
model = nn.Sequential(
    nn.Linear(64, 128),
    nn.LayerNorm(128),
    nn.Linear(128, 64),
)

optimizer = configure_optimizers(model, lr=1e-4, weight_decay=0.1)
print(f"参数组 0 (decay): {len(optimizer.param_groups[0]['params'])} 个参数")
print(f"参数组 1 (no decay): {len(optimizer.param_groups[1]['params'])} 个参数")

参数组 0 (decay): 2 个参数
参数组 1 (no decay): 4 个参数


## 4. 分布式优化器: ZeRO-2

nanochat 使用 ZeRO-2 风格的分布式 AdamW。

### 原理图解

```
传统 DDP (每个 GPU 都存完整状态):
┌─────────────────────────────────────────────────────┐
│ GPU 0: 完整参数 + 完整梯度 + 完整 m + 完整 v        │
│ GPU 1: 完整参数 + 完整梯度 + 完整 m + 完整 v        │
│ GPU 2: 完整参数 + 完整梯度 + 完整 m + 完整 v        │
│ GPU 3: 完整参数 + 完整梯度 + 完整 m + 完整 v        │
└─────────────────────────────────────────────────────┘

ZeRO-2 (分片优化器状态 + 梯度):
┌─────────────────────────────────────────────────────┐
│ GPU 0: 完整参数 + 1/4 梯度 + 1/4 m + 1/4 v         │
│ GPU 1: 完整参数 + 1/4 梯度 + 1/4 m + 1/4 v         │
│ GPU 2: 完整参数 + 1/4 梯度 + 1/4 m + 1/4 v         │
│ GPU 3: 完整参数 + 1/4 梯度 + 1/4 m + 1/4 v         │
└─────────────────────────────────────────────────────┘
```

### 通信模式

```
1. reduce_scatter: 每个 GPU 得到 1/N 梯度的平均
2. 各自更新自己负责的 1/N 参数
3. all_gather: 把更新后的参数片段合并
```

In [ ]:
# nanochat DistAdamW 核心逻辑 (简化版)

dist_adamw_pseudo = '''
def step(self):
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    
    for p in params:
        grad = p.grad
        
        # 1. reduce_scatter: 每个 rank 得到 1/N 的梯度
        rank_size = grad.shape[0] // world_size
        grad_slice = torch.empty_like(grad[:rank_size])
        dist.reduce_scatter_tensor(grad_slice, grad, op=ReduceOp.AVG)
        
        # 2. 只更新自己负责的 1/N 参数片段
        p_slice = p[rank * rank_size : (rank + 1) * rank_size]
        m_slice = self.state[p]['exp_avg']   # 只存 1/N
        v_slice = self.state[p]['exp_avg_sq'] # 只存 1/N
        
        # AdamW 更新 (标准公式)
        m_slice.mul_(beta1).add_(grad_slice, alpha=1-beta1)
        v_slice.mul_(beta2).addcmul_(grad_slice, grad_slice, value=1-beta2)
        ...
        p_slice.add_(update, alpha=-1)
        
        # 3. all_gather: 把各自更新的片段合并回完整参数
        dist.all_gather_into_tensor(p, p_slice)
'''
print(dist_adamw_pseudo)

### 显存节省计算

假设 4 个 GPU，1B 参数模型：

In [ ]:
num_params = 1e9
num_gpus = 4

# 传统 DDP
ddp_optimizer_mem = 12 * num_params  # m + v + master (每个 GPU 都存完整)

# ZeRO-2
zero2_optimizer_mem = 12 * num_params / num_gpus  # 分片

print(f"DDP 优化器显存 (每 GPU): {ddp_optimizer_mem / 1e9:.1f} GB")
print(f"ZeRO-2 优化器显存 (每 GPU): {zero2_optimizer_mem / 1e9:.1f} GB")
print(f"节省: {(1 - zero2_optimizer_mem/ddp_optimizer_mem)*100:.0f}%")

## 5. 学习率调度

### Warmup + Cosine Decay (标准配置)

```
学习率
  ↑
  │    /‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾\___
  │   /                          \_____
  │  /                                  \___
  └──┴────────────────────────────────────────→ Step
     ↑ Warmup                   Cosine Decay ↑
```

In [ ]:
def get_lr(step, warmup_steps, max_steps, max_lr, min_lr=0):
    """
    Warmup + Cosine Decay
    """
    # Warmup: 线性增加
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    
    # 结束后保持最小值
    if step >= max_steps:
        return min_lr
    
    # Cosine decay
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

# 可视化
import matplotlib.pyplot as plt

steps = range(1000)
lrs = [get_lr(s, warmup_steps=100, max_steps=1000, max_lr=1e-3, min_lr=1e-5) for s in steps]

plt.figure(figsize=(10, 4))
plt.plot(steps, lrs)
plt.axvline(x=100, color='r', linestyle='--', label='Warmup End')
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.title('Warmup + Cosine Decay')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 为什么需要 Warmup?

1. **Adam 的 bias correction**: 初始时 m, v 接近 0，更新不稳定
2. **batch norm 统计量**: 需要时间累积
3. **经验上**: 直接用大学习率容易发散

## 6. PyTorch 实现细节

### Fused AdamW

```python
# 普通版本: 多次 kernel launch
optimizer = torch.optim.AdamW(params, lr=1e-4)

# Fused 版本: 单次 kernel，更快
optimizer = torch.optim.AdamW(params, lr=1e-4, fused=True)
```

### foreach 优化

```python
# 批量处理多个 tensor，减少 kernel launch
optimizer = torch.optim.AdamW(params, lr=1e-4, foreach=True)
```

In [ ]:
# 性能对比 (示意)
print("AdamW 实现性能对比 (相对时间):")
print("  普通:     1.0x")
print("  foreach:  0.8x")
print("  fused:    0.6x  (推荐)")
print("\n注意: fused=True 需要 CUDA，且参数必须在同一设备")

## 7. 面试常见问题

### Q1: Adam 和 AdamW 的区别是什么?

**答**: Weight decay 的实现位置不同。
- Adam: L2 正则化加在 loss 里，梯度变成 `g + λθ`，会进入动量计算
- AdamW: Weight decay 直接作用于参数，`θ = θ - lr*λ*θ`，不污染动量

AdamW 在大模型训练中效果更好。

---

### Q2: 训练一个 7B 模型需要多少显存?

**答**: 混合精度训练约 16 * 参数量 字节。
- 7B * 16 bytes ≈ 112 GB (纯优化器+参数+梯度)
- 加上激活值，单卡 A100-80G 装不下
- 需要用 ZeRO、Tensor Parallel 等技术

---

### Q3: ZeRO-1/2/3 分别优化了什么?

**答**:
- ZeRO-1: 分片优化器状态 (m, v)
- ZeRO-2: + 分片梯度
- ZeRO-3: + 分片模型参数

通信开销: ZeRO-1 < ZeRO-2 < ZeRO-3

---

### Q4: 为什么 Embedding 和 LayerNorm 不加 weight decay?

**答**:
- Embedding: 稀疏访问，低频词会被过度惩罚
- LayerNorm: 是缩放参数，衰减会破坏归一化效果
- Bias: 维度太小，正则化收益可忽略

---

### Q5: 为什么需要 learning rate warmup?

**答**:
1. Adam 的 bias correction 在初始步不稳定
2. 模型初始化时梯度方向噪声大
3. 大学习率 + 不稳定梯度 = 发散

经验值: warmup_steps = 总步数的 1-5%

---

### Q6: 优化器状态需要保存到 checkpoint 吗?

**答**: 如果需要断点续训，**必须保存**。
- 保存: m, v, step 计数
- 不保存: 恢复后动量从 0 开始，需要重新 warmup

```python
# 保存
torch.save({
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),  # 包含 m, v
    'step': step,
}, 'checkpoint.pt')
```

## 8. 总结速查表

| 主题 | 要点 |
|------|------|
| **显存占用** | 混合精度: ~16Φ bytes (参数+梯度+优化器状态) |
| **参数分组** | 1D 参数 (bias/norm) 不加 weight decay |
| **分布式** | ZeRO-2: reduce_scatter → 本地更新 → all_gather |
| **学习率** | Warmup (1-5%) + Cosine Decay |
| **PyTorch** | 用 `fused=True` 提速 |
| **Checkpoint** | 必须保存 optimizer.state_dict() |

### nanochat 使用的配置

```python
# nanochat/gpt.py
adamw_kwargs = dict(
    betas=(0.8, 0.95),  # 注意: β1 比默认值 0.9 小
    eps=1e-10,
    weight_decay=0.0,   # embedding 不用 decay
)
```

## 参考资料

- [Decoupled Weight Decay (AdamW)](https://arxiv.org/abs/1711.05101)
- [ZeRO: Memory Optimizations](https://arxiv.org/abs/1910.02054)
- nanochat/adamw.py - 分布式 AdamW 实现